In [ ]:
import requests
import pandas as pd
from datetime import datetime, timedelta

import xgboost as xgb
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [ ]:
# --- GLOBÁLNÍ PARAMETRY (Definováno outside)
LAT = 50.0755
LON = 14.4378

URL = 'https://archive-api.open-meteo.com/v1/era5'

# Srape data

In [97]:
def download_weather_data(years_back, lat, lon, endpoint):
    """
    Stáhne historická weather data a připraví je pro XGBoost.
    """
    # 1. VÝPOČET DATUMŮ
    end_date_dt = datetime.now()
    start_date_dt = end_date_dt - timedelta(days=years_back * 365)
    
    start_date = start_date_dt.date().isoformat()
    end_date = end_date_dt.date().isoformat()

    # Metadata (Features)
    variables = [
        'temperature_2m', 
        'surface_pressure', 
        'relative_humidity_2m', 
        'precipitation', 
        'cloud_cover', 
        'wind_speed_10m'
    ]

    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": ",".join(variables),
        "timezone": "Europe/Berlin"
    }

    print(f"Stahuji data z: {endpoint}")
    print(f"Lokalita: {lat}, {lon} | Rozsah: {start_date} až {end_date}")

    # 2. VOLÁNÍ API
    try:
        response = requests.get(endpoint, params=params)
        response.raise_for_status() 
        
        data = response.json()
        df = pd.DataFrame(data.get('hourly', {}))
        
        # 3. FEATURE ENGINEERING
        df['time'] = pd.to_datetime(df['time'])
        df['hour'] = df['time'].dt.hour
        df['day_of_week'] = df['time'].dt.dayofweek
        df['month'] = df['time'].dt.month
        df['day_of_year'] = df['time'].dt.dayofyear
        df['year'] = df['time'].dt.year
        
        file_name = f"in/weather_data_{years_back}y.csv"
        df.to_csv(file_name, index=False)
        
        print(f"Hotovo! Data uložena do: {file_name}")
        return df

    except Exception as e:
        print(f"Nastala chyba při stahování: {e}")
        return None

In [98]:
# 2. Volání funkce
df_history = download_weather_data(
    years_back=10, 
    lat=LAT, 
    lon=LON, 
    endpoint=URL
)

Stahuji data z: https://archive-api.open-meteo.com/v1/era5
Lokalita: 50.0755, 14.4378 | Rozsah: 2016-02-09 až 2026-02-06
Hotovo! Data uložena do: in/weather_data_10y.csv


# Feature engineering

In [283]:
df = pd.read_csv('in/weather_data_10y.csv')
df.head(5)

,time,temperature_2m,surface_pressure,relative_humidity_2m,precipitation,cloud_cover,wind_speed_10m,hour,day_of_week,month,day_of_year,year
0,2016-02-09 00:00:00,6.1,974.1,75,0.0,57,27.4,0,1,2,40,2016
1,2016-02-09 01:00:00,5.7,974.0,77,0.0,52,26.7,1,1,2,40,2016
2,2016-02-09 02:00:00,5.5,974.1,79,0.0,60,26.7,2,1,2,40,2016
3,2016-02-09 03:00:00,5.6,974.4,76,0.0,91,27.7,3,1,2,40,2016
4,2016-02-09 04:00:00,5.7,974.6,74,0.0,49,27.2,4,1,2,40,2016


### Add features

In [292]:
categorical_features = ['hour', 
                        #'day_of_week', 
                        #'day_of_year',
                        'month', 
                        'year'
                        ]

for col in categorical_features:
    df[col] = df[col].astype('category')


df.head(5)

,time,temperature_2m,surface_pressure,relative_humidity_2m,precipitation,cloud_cover,wind_speed_10m,hour,day_of_week,month,day_of_year,year
0,2016-02-09 00:00:00,6.1,974.1,75,0.0,57,27.4,0,1,2,40,2016
1,2016-02-09 01:00:00,5.7,974.0,77,0.0,52,26.7,1,1,2,40,2016
2,2016-02-09 02:00:00,5.5,974.1,79,0.0,60,26.7,2,1,2,40,2016
3,2016-02-09 03:00:00,5.6,974.4,76,0.0,91,27.7,3,1,2,40,2016
4,2016-02-09 04:00:00,5.7,974.6,74,0.0,49,27.2,4,1,2,40,2016


In [293]:
# Teď definujeme tvůj původní seznam + nové rysy
features = ['surface_pressure', 
            'relative_humidity_2m', 
            'cloud_cover', 
            'wind_speed_10m', 
            'hour', 
            #'day_of_week', 
            'month', 
            #'day_of_year',
            'year'
            ]

target = 'temperature_2m' 

In [294]:
df.dtypes

time                      object
temperature_2m           float64
surface_pressure         float64
relative_humidity_2m       int64
precipitation            float64
cloud_cover                int64
wind_speed_10m           float64
hour                    category
day_of_week                int64
month                   category
day_of_year                int64
year                    category
dtype: object

In [295]:
X = df[features]
y = df[target]

# Rozdělení na trénovací a testovací sadu (vezmeme posledních 500 hodin jako test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=8000, shuffle=False)

# 2. TRÉNOVÁNÍ KVANTILŮ (10%, 50%, 90%)
quantiles = [0.1, 0.5, 0.9]
preds = {}

# Model

In [296]:
for q in quantiles:
    print(f"Trénuji model pro kvantil: {q}")
    model = xgb.XGBRegressor(
        objective='reg:quantileerror', # Nastavení pro kvantilovou regresi
        quantile_alpha=q,
        n_estimators=100,
        max_depth=4,
        learning_rate=0.1,
        enable_categorical=True
    )
    model.fit(X_train, y_train)
    preds[q] = model.predict(X_test)


Trénuji model pro kvantil: 0.1
Trénuji model pro kvantil: 0.5
Trénuji model pro kvantil: 0.9


# Quality

In [297]:
# Spočítáme, kolikrát realita padla do intervalu
hits = (y_test >= preds[0.1]) & (y_test <= preds[0.9])
coverage = hits.mean()
avg_width = (preds[0.9] - preds[0.1]).mean()

print(f"--- VERDIKT O MODELU ---")
print(f"Spolehlivost (Coverage): {coverage:.1%} (Ideál pro tvé kvantily je 80%)")
print(f"Průměrná nejistota: {avg_width:.2f} °C")

if coverage < 0.75:
    print("VAROVÁNÍ: Model je příliš sebevědomý a často se plete!")
elif coverage > 0.85:
    print("VAROVÁNÍ: Model je příliš opatrný a dává zbytečně široké pásmo.")
else:
    print("GRATULACE: Model je skvěle zkalibrovaný.")

--- VERDIKT O MODELU ---
Spolehlivost (Coverage): 64.2% (Ideál pro tvé kvantily je 80%)
Průměrná nejistota: 7.07 °C
VAROVÁNÍ: Model je příliš sebevědomý a často se plete!


# Plot the charts

In [298]:
import plotly.graph_objects as go
import pandas as pd

def plot_quantile_predictions(y_true, predictions, time_index, low_q=0.1, high_q=0.9, title="Kvantilová regrese"):
    """
    Vytvoří interaktivní Plotly graf s datem na ose X.
    y_true: skutečné hodnoty (y_test)
    predictions: slovník s predikcemi
    time_index: časová osa (např. df_test['time'])
    """
    
    # 1. Příprava DataFrame se skutečným časem
    plot_df = pd.DataFrame({
        'Time': time_index,
        'Actual': y_true.values if hasattr(y_true, 'values') else y_true,
        'Low': predictions[low_q],
        'Median': predictions[0.5],
        'High': predictions[high_q]
    })

    fig = go.Figure()

    # Spodní hranice (neviditelná)
    fig.add_trace(go.Scatter(
        x=plot_df['Time'], y=plot_df['Low'],
        mode='lines', line=dict(width=0),
        showlegend=False, name=f'Q{low_q}'
    ))

    # Horní hranice + Vyplnění
    fig.add_trace(go.Scatter(
        x=plot_df['Time'], y=plot_df['High'],
        mode='lines', line=dict(width=0),
        fill='tonexty', 
        fillcolor='rgba(255, 165, 0, 0.3)', 
        name=f'{int((high_q - low_q)*100)}% Interval spolehlivosti'
    ))

    # Medián (Předpověď)
    fig.add_trace(go.Scatter(
        x=plot_df['Time'], y=plot_df['Median'],
        mode='lines', line=dict(color='red', dash='dash'),
        name='Medián (Předpověď)'
    ))

    # Skutečnost
    fig.add_trace(go.Scatter(
        x=plot_df['Time'], y=plot_df['Actual'],
        mode='lines', line=dict(color='black', width=2),
        name='Skutečná hodnota'
    ))

    # 3. Úprava vzhledu - přidáme Range Slider pro pohodlné listování časem
    fig.update_layout(
        title=f'{title} ({low_q} - {high_q})',
        xaxis_title='Datum a čas',
        yaxis_title='Hodnota',
        template='plotly_white',
        hovermode='x unified'
    )
    
    # Range slider umožní "zoomovat" v čase jako na burze
    fig.update_xaxes(rangeslider_visible=True)

    return fig

In [299]:
# Nejdříve vytáhneme časové údaje, které odpovídají tvým testovacím datům
# iloc[y_test.index] zajistí, že časy budou přesně sedět na ty řádky, které model předpovídal
time_test = df['time'].loc[y_test.index] 

# Teď už můžeš volat funkci, protože time_test už existuje
fig = plot_quantile_predictions(
    y_true=y_test, 
    predictions=preds, 
    time_index=time_test, 
    low_q=0.1, 
    high_q=0.9, 
    title="Předpověď počasí Praha"
)

fig.show()

# Predict future

In [75]:
# Seznam features bez lagů (přesně jak jsi chtěl)
features = ['surface_pressure', 'relative_humidity_2m', 'cloud_cover', 
            'wind_speed_10m', 'hour', 'day_of_week', 'month', 'day_of_year']

final_models = {}

for q in [0.1, 0.5, 0.9]:
    print(f"Trénuji finální model pro kvantil {q}...")
    # Použijeme tvé ověřené parametry
    m = xgb.XGBRegressor(objective='reg:quantileerror', quantile_alpha=q)
    m.fit(X, y) # X a y jsou z tvého celého vyčištěného datasetu
    final_models[q] = m

Trénuji finální model pro kvantil 0.1...
Trénuji finální model pro kvantil 0.5...
Trénuji finální model pro kvantil 0.9...


In [ ]:
# Vytvoříme prázdný DataFrame pro výsledky
forecast_results = pd.DataFrame({'time': df_future['time']})

# Proženeme budoucí data našimi modely
for q in [0.1, 0.5, 0.9]:
    forecast_results[f'q_{q}'] = final_models[q].predict(df_future[features])

# Přejmenujeme pro lepší čitelnost
forecast_results.columns = ['Čas', 'Spodní hranice (10%)', 'Odhad (50%)', 'Horní hranice (90%)']

display(forecast_results)